# Products, quotients, and symmetric powers

This tutorial explains the constructions behind `fastop`'s compact quotient models and shows the recommended route from a finite simplicial set to $\operatorname{Sym}^n(X)=X^n/S_n$.


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / 'src' / 'fastop').exists():
    for candidate in (repo.parent, repo.parent.parent):
        if (candidate / 'src' / 'fastop').exists():
            repo = candidate
            break
sys.path.insert(0, str(repo / 'src'))

from fastop import DeltaComplex, FiniteGroupAction, SimplicialSet, spaces


## 1. Cartesian products of simplicial sets

Products must retain degeneracies until all factors have been normalized together. `SimplicialSet.cartesian_product` performs this bookkeeping and stores enough information to permute equal factors later.


In [ ]:
circle = SimplicialSet.sphere(1)
torus = circle.cartesian_product(circle)
print('circle:', circle.f_vector())
print('product torus:', torus.f_vector())
assert torus.cohomology(p=3).betti_numbers() == {0: 1, 1: 2, 2: 1}


## 2. Strict finite actions on face maps

A group-action generator is a permutation of the cells in every degree that commutes with every face map. Here a cyclic group rotates a polygonal circle. The quotient has one vertex and one loop edge.


In [ ]:
order = 5
polygon = DeltaComplex([
    [() for _ in range(order)],
    [((i + 1) % order, i) for i in range(order)],
])
rotation = FiniteGroupAction.from_cell_maps(
    polygon,
    lambda degree, cell: (cell + 1) % order,
)
assert rotation.order(polygon) == order
assert rotation.is_free(polygon)
quotient_circle = polygon.quotient(rotation, require_free=True)
assert quotient_circle.f_vector() == (1, 1)


Use `require_free=True` only when freeness is mathematically expected. It enumerates the generated finite group and rejects fixed cells. Quotients with stabilizers are still valid Delta-complexes in many cases, but their interpretation needs more care.


## 3. Factor permutations and symmetric quotients

For a small diagnostic example, one may explicitly construct $X^n$, ask for the factor-permutation action, and quotient. For real computations, `symmetric_power(n)` is preferable because it constructs compact unordered labels directly.


In [ ]:
sphere = SimplicialSet.sphere(2)
square = sphere.cartesian_product(sphere)
transposition = FiniteGroupAction.cyclic(
    square.factor_permutation_action((1, 0))
)
explicit_sym2 = square.quotient(transposition)
direct_sym2 = sphere.symmetric_power(2)

assert explicit_sym2.f_vector() == direct_sym2.f_vector()
assert direct_sym2.cohomology(p=2).betti_numbers() == {0: 1, 2: 1, 4: 1}
print('Sym^2(S^2):', direct_sym2.f_vector())


## 4. Always preflight a large symmetric power

`symmetric_power_f_vector(n)` uses inclusion–exclusion to count nondegenerate cells without constructing their labels. The symmetric-power object is degree-lazy, but an individual boundary layer may still be large.


In [ ]:
surface = spaces.orientable_surface(1)
for power in (2, 3, 5, 7):
    predicted = surface.symmetric_power_f_vector(power)
    print(f'Sym^{power}(T^2): {sum(predicted):,} cells')


The count is a safety gate, not merely a statistic. Construct only models whose relevant degree layers fit the available memory. Calling `f_vector()` on a direct symmetric power remains cheap; calling `cells(d)` materializes degree $d$.


## 5. Surface-family constructors

Both oriented and non-orientable surfaces use compact one-vertex models and the same symmetric-power engine.


In [ ]:
sym3_torus = spaces.orientable_surface(1).symmetric_power(3)
sym3_rp2 = spaces.nonorientable_surface(1).symmetric_power(3)

assert sym3_torus.cohomology(p=3).operation_rank(2, 1) == 1
assert sym3_rp2.cohomology(p=3).betti_numbers() == {0: 1}
assert sym3_rp2.cohomology(p=2).operation_rank(2, 2) == 1


The scientific notebooks `symmetric_products_of_surfaces.ipynb` and `lens_spaces.ipynb` develop the larger examples and their mathematical ground truth.
